# Modric Corner-Kicks Analysis
## AC Milan 2025/2026 Serie A Season

This notebook analyzes all corner-kicks taken by Luka Modrić during the 2025/2026 Serie A season with AC Milan, including their outcomes and effectiveness.

## 1. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import glob
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

## 2. Load and Explore CSV Event Data Files

In [2]:
# Define the data directory for 2025/2026 season
data_dir = Path("../data/raw/serie_a_2025_2026/events")
event_files = sorted(data_dir.glob("*.csv"))

print(f"Total event files found: {len(event_files)}")
print(f"\nData directory: {data_dir}")
print(f"\nSample files:")
for f in event_files[:5]:
    print(f"  - {f.name}")

# Load and inspect one sample file to understand the structure
sample_file = event_files[0]
sample_df = pd.read_csv(sample_file, low_memory=False)

print(f"\n\nSample file columns ({len(sample_df.columns)} total):")
print(sample_df.columns.tolist()[:20])

print(f"\nSample file shape: {sample_df.shape}")
print(f"Sample rows:")
print(sample_df[['event', 'type_id', 'team_name', 'player_name', 'x', 'y']].head(10))

Total event files found: 309

Data directory: ../data/raw/serie_a_2025_2026/events

Sample files:
  - 10_Cremonese_Juventus_digmzny5no6kxh6hfopalgahg.csv
  - 10_Fiorentina_Lecce_diuxkzq1t6j5r3b8myu04i3h0.csv
  - 10_Lazio_Cagliari_dj93wlc4y6a5z30ns89tmayac.csv
  - 10_Milan_Roma_djnhnvs0hjjv7p1rlkt1eizh0.csv
  - 10_Napoli_Como_dk1y2p3cuhqk9g9jrvfmk6784.csv


Sample file columns (248 total):
['general_id', 'event_id', 'event', 'type_id', 'period_id', 'time_min', 'time_sec', 'contestant_id', 'team_name', 'team_code', 'team_position', 'player_id', 'player_name', 'x', 'y', 'outcome', 'timeStamp', 'lastModified', 'match_id', 'coverage_level']

Sample file shape: (1705, 248)
Sample rows:
          event  type_id     team_name     player_name     x     y
0  Team setp up       34  US Cremonese             NaN   0.0   0.0
1  Team setp up       34   Juventus FC             NaN   0.0   0.0
2         Start       32  US Cremonese             NaN   0.0   0.0
3         Start       32   Juventus FC     

## 3. Filter Milan Games from 2025/2026 Season

In [3]:
# Find all Milan match files
milan_files = [f for f in event_files if 'Milan' in f.name]
print(f"AC Milan matches found: {len(milan_files)}\n")

# Load all Milan match data
milan_data = []
milan_matches = []

for file_path in milan_files:
    try:
        df = pd.read_csv(file_path, low_memory=False)
        # Extract match identifier from filename
        filename = file_path.name
        parts = filename.split('_')
        
        # Extract gameweek (first part)
        match_id = parts[0]
        
        # Get unique teams from the data
        teams = df['team_name'].unique()
        teams_clean = [t for t in teams if pd.notna(t) and t.strip()]
        
        # Identify Milan and opponent
        # Use exact matching for AC Milan, not substring
        milan_team = None
        opponent_team = None
        for team in teams_clean:
            if team == 'AC Milan':
                milan_team = team
            else:
                opponent_team = team
        
        # Determine side
        if 'Milan_' in filename:
            milan_side = 'home'
        else:
            milan_side = 'away'
        
        opponent_name = opponent_team if opponent_team else 'Unknown'
        
        # Fix for Inter if still showing Unknown
        if opponent_name == 'Unknown' and 'Inter' in filename:
            opponent_name = 'F.C. Internazionale'
        
        milan_matches.append({
            'match_file': file_path.name,
            'gameweek': match_id,
            'opponent': opponent_name,
            'milan_side': milan_side,
            'rows': len(df)
        })
        
        milan_data.append(df)
    except Exception as e:
        print(f"Error loading {file_path.name}: {e}")

# Create a combined dataframe with match metadata
all_milan_events = []
for i, df in enumerate(milan_data):
    df['_file_index'] = i
    all_milan_events.append(df)

milan_combined = pd.concat(all_milan_events, ignore_index=True)

print(f"Total events across all Milan matches: {len(milan_combined)}")
print(f"\nMilan matches:")
for m in milan_matches:
    print(f"  GW{m['gameweek']}: vs {m['opponent']} ({m['milan_side']}) - {m['rows']} events")

AC Milan matches found: 31

Total events across all Milan matches: 52405

Milan matches:
  GW10: vs AS Roma (home) - 1678 events
  GW11: vs Parma Calcio 1913 (home) - 1688 events
  GW12: vs FC Internazionale Milano (home) - 1767 events
  GW13: vs SS Lazio (home) - 1792 events
  GW14: vs Torino FC (home) - 1614 events
  GW15: vs US Sassuolo Calcio (home) - 1717 events
  GW16: vs Calcio Como 1907 (home) - 1735 events
  GW17: vs Hellas Verona FC (home) - 1574 events
  GW18: vs Cagliari Calcio (home) - 1782 events
  GW19: vs Genoa CFC (home) - 1777 events
  GW1: vs US Cremonese (home) - 1747 events
  GW20: vs ACF Fiorentina (home) - 1583 events
  GW21: vs US Lecce (home) - 1766 events
  GW22: vs AS Roma (home) - 1754 events
  GW23: vs Bologna FC 1909 (home) - 1709 events
  GW24: vs Calcio Como 1907 (home) - 1681 events
  GW25: vs Pisa Sporting Club (home) - 1723 events
  GW26: vs Parma Calcio 1913 (home) - 1779 events
  GW27: vs US Cremonese (home) - 1678 events
  GW28: vs FC Internazional

## 4. Identify Corner Kicks Taken by Modrić

In [4]:
# Look for corner events in the combined data
corner_events = milan_combined[milan_combined['event'].str.contains('corner', case=False, na=False)]
print(f"Total corner events found: {len(corner_events)}")
print(f"Unique corner event types: {corner_events['event'].unique()}")

# Check all unique players to see if Modric is in the data
# IMPORTANT: Filter for AC Milan ONLY (not F.C. Internazionale which also contains "Milan")
print(f"\n\nChecking for Modric in AC Milan players...")
milan_events = milan_combined[milan_combined['team_name'] == 'AC Milan']
print(f"Total AC Milan events: {len(milan_events)}")

unique_players = milan_events['player_name'].dropna().unique()
print(f"\nUnique AC Milan players: {len(unique_players)}")

# Look for players with similar names
modric_like = [p for p in unique_players if p and 'odric' in p.lower()]
print(f"\nPlayers with 'odric' in name: {modric_like}")

# Check for all players
print(f"\nAll AC Milan players:")
for i, p in enumerate(sorted(unique_players)):
    print(f"  {p}")

# Let's look for the corners awarded to AC Milan and see who took them
print(f"\n\nCorners awarded to AC Milan:")
milan_corners_awarded = corner_events[corner_events['team_name'] == 'AC Milan']
print(f"Total: {len(milan_corners_awarded)}")

if len(milan_corners_awarded) > 0:
    print(f"\nSample corners awarded to AC Milan:")
    print(milan_corners_awarded[['time_min', 'time_sec', 'team_name', 'player_name', 'x', 'y']].head(10))

Total corner events found: 560
Unique corner event types: ['Corner Awarded']


Checking for Modric in AC Milan players...
Total AC Milan events: 26968

Unique AC Milan players: 26

Players with 'odric' in name: []

All AC Milan players:
  A. Jashari
  A. Rabiot
  A. Saelemaekers
  C. Balentien
  C. Nkunku
  C. Pulišić
  D. Bartesaghi
  D. Odogu
  F. Tomori
  K. De Winter
  L. Modrić
  M. Gabbia
  M. Maignan
  N. Füllkrug
  P. Estupiñán
  P. Terracciano
  R. Loftus-Cheek
  Rafael Leão
  S. Chukwueze
  S. Giménez
  S. Pavlović
  S. Ricci
  Y. Fofana
  Y. Musah
  Z. Athekame
  Álex Jiménez


Corners awarded to AC Milan:
Total: 280

Sample corners awarded to AC Milan:
      time_min  time_sec team_name      player_name     x     y
405         19        20  AC Milan  A. Saelemaekers  95.8  16.7
474         23         0  AC Milan    D. Bartesaghi  95.6  90.2
549         26        19  AC Milan      Rafael Leão  96.0  68.2
717         36        59  AC Milan      Rafael Leão  92.2  60.9
970    

## 5. Analyze Corner Kick Outcomes

In [5]:
# Filter for corner kicks taken by Modric
# These are passes immediately after "Corner Awarded" events where Modric is the player
# OR corner awarded events where Modric is listed

print("Finding Modric's corner kicks...")

# Method 1: Look for Corner Awarded events with Modric as player
modric_corners_direct = corner_events[
    (corner_events['player_name'] == 'L. Modrić') &
    (corner_events['team_name'] == 'AC Milan')
]

# Method 2: Look for passes immediately after Corner Awarded events by Modric
modric_corners_via_pass = []
for idx, corner_row in milan_corners_awarded[
    (milan_corners_awarded['player_name'] == 'L. Modrić')
].iterrows():
    file_idx = int(corner_row['_file_index'])
    file_df = milan_data[file_idx]
    corner_time = corner_row['time_min'] * 60 + corner_row['time_sec']
    
    # Look for Modric's passes within a few seconds after the corner
    modric_events = file_df[
        (file_df['player_name'] == 'L. Modrić') &
        (file_df['time_min'] * 60 + file_df['time_sec'] > corner_time) &
        (file_df['time_min'] * 60 + file_df['time_sec'] <= corner_time + 5) &
        (file_df['event'] == 'Pass')
    ]
    
    if len(modric_events) > 0:
        modric_corners_via_pass.append({
            'type': 'corner_pass',
            'row': modric_events.iloc[0],
            'file_idx': file_idx,
            'corner_event': corner_row
        })

print(f"Corner Awarded events to Modric: {len(modric_corners_direct)}")
print(f"Passes by Modric after corners: {len(modric_corners_via_pass)}")

# Use the corner awarded events as our source of truth
modric_corners = milan_corners_awarded[
    (milan_corners_awarded['player_name'] == 'L. Modrić')
].copy()

print(f"\nTotal corner kicks taken by Modric: {len(modric_corners)}")

if len(modric_corners) > 0:
    print(f"\nModric's corner kick events:")
    print(modric_corners[['time_min', 'time_sec', 'team_name', 'player_name', 'outcome', 'x', 'y']].to_string())

Finding Modric's corner kicks...
Corner Awarded events to Modric: 18
Passes by Modric after corners: 0

Total corner kicks taken by Modric: 18

Modric's corner kick events:
       time_min  time_sec team_name player_name  outcome     x     y
4885         80        34  AC Milan   L. Modrić        0   0.8  27.4
12752        45         7  AC Milan   L. Modrić        1  95.8  27.0
14141        27        23  AC Milan   L. Modrić        1  88.6  24.6
18167        51        48  AC Milan   L. Modrić        1  91.6  50.2
22925        35        45  AC Milan   L. Modrić        1  91.0  51.6
24058         3         3  AC Milan   L. Modrić        0   9.0  27.0
28271        45        41  AC Milan   L. Modrić        0   3.0   4.9
35732        77        41  AC Milan   L. Modrić        1  98.6  28.6
35973        92        26  AC Milan   L. Modrić        1  87.9  49.0
40901        93        43  AC Milan   L. Modrić        1  98.5  73.7
41077         7        10  AC Milan   L. Modrić        1  84.2  66.2

## 6. Create Summary Table

In [6]:
# Improved function to track outcome of each corner kick
def analyze_corner_outcome(corner_row, file_idx, all_files_data, milan_matches_list):
    """
    Analyze what happens after a corner kick is taken by Modric.
    Tracks the ball for events after the corner to determine outcome.
    """
    match_file_df = all_files_data[file_idx]
    
    time_minute = corner_row['time_min']
    time_second = corner_row['time_sec']
    corner_time = time_minute * 60 + time_second
    
    # Look at events in next 30 seconds after the corner
    subsequent = match_file_df[
        (match_file_df['time_min'] * 60 + match_file_df['time_sec'] > corner_time) &
        (match_file_df['time_min'] * 60 + match_file_df['time_sec'] <= corner_time + 30)
    ].reset_index(drop=True)
    
    outcome_type = 'Out of play'
    outcome_detail = ''
    reached_team = 'Neither'
    
    # Analyze subsequent events
    for i, event_row in subsequent.iterrows():
        event_type = str(event_row.get('event', '')).lower()
        player = event_row.get('player_name', '')
        team = event_row.get('team_name', '')
        
        # Skip if no player info
        if not player or pd.isna(player):
            continue
        
        # Check for goal
        if 'goal' in event_type:
            outcome_type = 'GOAL'
            outcome_detail = f"Goal by {player}"
            reached_team = 'Milan'
            break
        
        # Check if Milan player touched the ball
        if 'Milan' in team:
            if reached_team != 'Milan':
                outcome_type = 'Milan player touch'
                reached_team = 'Milan'
                outcome_detail = f"First touch: {player}"
            
            # Check for shot
            if 'shot' in event_type or 'miss' in event_type:
                outcome_type = 'Shot'
                outcome_detail = f"Shot by {player}"
                break
        
        # Check if opponent touched the ball
        elif 'Milan' not in team and reached_team == 'Neither':
            outcome_type = 'Opponent touch'
            reached_team = 'Opponent'
            outcome_detail = f"First touch: {player} ({team})"
        
        # If it goes out or other end events
        if event_type in ['out', 'end']:
            break
    
    return {
        'outcome_type': outcome_type,
        'outcome_detail': outcome_detail,
        'reached': reached_team
    }

# Analyze all Modric corners with correct data
corner_analyses = []

for idx, corner_row in modric_corners.iterrows():
    file_idx = int(corner_row['_file_index'])
    match_info = milan_matches[file_idx]
    
    outcome = analyze_corner_outcome(corner_row, file_idx, milan_data, milan_matches)
    
    corner_analyses.append({
        'Gameweek': match_info['gameweek'],
        'Opponent': match_info['opponent'],
        'Minute': f"{int(corner_row['time_min'])}:{int(corner_row['time_sec']):02d}",
        'Outcome Type': outcome['outcome_type'],
        'Outcome Detail': outcome['outcome_detail'],
        'Reached': outcome['reached'],
        'Result': corner_row['outcome']
    })

print(f"Analyzed {len(corner_analyses)} corner kicks by Modric\n")

Analyzed 18 corner kicks by Modric



In [7]:
# Create the summary dataframe
corners_df = pd.DataFrame(corner_analyses)

print("=" * 130)
print("LUKA MODRIĆ CORNER KICKS SUMMARY - AC MILAN 2025/2026 SERIE A SEASON")
print("=" * 130)
print(f"\nTotal Corner Kicks by Modrić: {len(corners_df)}\n")

# Display full table
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print(corners_df.to_string(index=False))

# Calculate statistics
print("\n" + "=" * 130)
print("OUTCOME STATISTICS")
print("=" * 130)

outcome_counts = corners_df['Outcome Type'].value_counts()
print("\nOutcomes Breakdown:")
for outcome, count in outcome_counts.items():
    pct = (count / len(corners_df)) * 100
    print(f"  • {outcome}: {count} ({pct:.1f}%)")

print("\nReached Statistics:")
reached_counts = corners_df['Reached'].value_counts()
for reached, count in reached_counts.items():
    pct = (count / len(corners_df)) * 100
    print(f"  • {reached}: {count} ({pct:.1f}%)")

print("\nOpponents Faced:")
opponent_counts = corners_df['Opponent'].value_counts()
for opp, count in opponent_counts.items():
    print(f"  • vs {opp}: {count} corner{'s' if count > 1 else ''}")

print("\n" + "=" * 130)

LUKA MODRIĆ CORNER KICKS SUMMARY - AC MILAN 2025/2026 SERIE A SEASON

Total Corner Kicks by Modrić: 18

Gameweek                   Opponent Minute       Outcome Type                                          Outcome Detail  Reached  Result
      12   FC Internazionale Milano  80:34               Shot                                       Shot by F. Acerbi    Milan       0
      17           Hellas Verona FC  45:07               Shot                                       Shot by A. Rabiot    Milan       1
      18            Cagliari Calcio  27:23        Out of play                                                          Neither       1
       1               US Cremonese  51:48               Shot                                      Shot by C. Pulišić    Milan       1
      22                    AS Roma  35:45 Milan player touch                              First touch: D. Bartesaghi    Milan       1
      23            Bologna FC 1909   3:03 Milan player touch                         